In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10.95})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)
figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)
import jax.numpy as jnp
import equinox as eqx 

In [ ]:
from admpe.utils.helper import pprint_helper
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results, load_modeling_result, provide_modeling_results_overview, load_excitation_result
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result,
    evaluate_model_prediction, evaluate_model_rollout, evaluate_persistence_prediction
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts

---

## update df:

In [ ]:
from admpe.data_management import QUANTITATIVE_RESULTS_ROOT

In [ ]:
# exp_name = "noiseless_w_obj"
# exp_name = "noise_1_e_minus_6"
# exp_name = "state_only_run1"
# exp_name = "state_only_pm_no_noise"
#exp_name = "state_only_pm_no_noise"
# exp_name = "state_only_gru"
# exp_name = "state_only_gru_quick"
# exp_name = "submission_nov"

exp_name = "submission_nov_curv64"

# exp_name = "state_only_pm_no_noise_gru_testing_validation"
# exp_name = "pm_no_noise_gru"
# exp_name = "state_only_pm_no_noise_one_step"

filter_testing_points = True
tag = "safely_excitable" if filter_testing_points else "full_box"

load_if_available = True
show_baseline = False

In [ ]:
modeling_results_paths = list_modeling_results(
    system_name="pmsm_nominal",
    model_type="gru",
    status="completed",
    exp_name=exp_name,
    excitation_objective=[
        "dmpe_static_curv",
        "dmpe_static_mixture_curv",
    ],
)
print(f"{len(modeling_results_paths)} completed models")

# modeling_results_paths = list_modeling_results(
#     system_name=None, #"cart_pole",
#     model_type="gru", #"neural_euler_large_1", #"gru", # 
#     status="completed",
#     exp_name=exp_name,
#     excitation_objective=None,
    
# )
# print(len(modeling_results_paths))
# modeling_results_paths

In [ ]:
import logging
logging.basicConfig()
logging.getLogger().setLevel(logging.WARNING)

name = f"modeling_results_{exp_name}_{tag}"

modeling_results = None

if (QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet").exists() and load_if_available:
    modeling_results = pd.read_parquet(QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet")

modeling_results = update_modeling_df(modeling_results, modeling_results_paths, filter_testing_points=filter_testing_points)
modeling_results.to_parquet(QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet")
print("Updated df stored to: ", QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet")

In [ ]:
modeling_results = pd.read_parquet(QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet")
modeling_results

## Get persistence results as baseline:

In [ ]:
if show_baseline:
    persistence_results = {
        system_name: evaluate_persistence_prediction(
            system_name,
            filter_testing_points=filter_testing_points,
        )
        for system_name in modeling_results["system_name"].dropna().unique()
    }
    pprint_helper(persistence_results)

## Model performance by excitation objective

Each system has one figure with RMSE, MAE and WCE panels. Boxes group individual runs by model and excitation objective; lower errors are better. The sample counts exclude missing or infinite metric values. Logarithmic axes make the wide error ranges easier to compare; set `log_scale=False` for linear axes.

In [ ]:
from admpe.evaluation.modeling_visualization import visualize_model_prediction_metric_boxplots, visualize_model_rollout_metric_boxplots

In [ ]:
metric_boxplots = visualize_model_prediction_metric_boxplots(modeling_results, log_scale=True, figure_width=figurewidth_in, hide_outliers=False)

figure_root = QUANTITATIVE_RESULTS_ROOT / "modeling_figs" / "boxplots"
figure_root.mkdir(parents=True, exist_ok=True)

for system_name, (fig, axes) in metric_boxplots.items():
    if show_baseline:
        baseline_metrics = persistence_results[system_name]["metrics"]
        for ax, metric in zip(axes, ("rmse", "mae", "wce")):
            ax.axhline(
                baseline_metrics[metric],
                color="black",
                linestyle="--",
                linewidth=1.5,
                label="Baseline",
                zorder=3,
            )
        axes[0].legend(loc="best")

    fig.savefig(figure_root / f"{name}-boxplots-pred-{system_name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(figure_root / f"{name}-boxplots-pred-{system_name}.pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)

print("Figures saved to:", figure_root)

In [ ]:
rollout_boxplots = visualize_model_rollout_metric_boxplots(
    modeling_results,
    log_scale=True,
    figure_width=figurewidth_in,
    hide_outliers=False,
)
figure_root = QUANTITATIVE_RESULTS_ROOT / "modeling_figs" / "boxplots"
figure_root.mkdir(parents=True, exist_ok=True)

for (system_name, signal_name), (fig, axes) in rollout_boxplots.items():
    fig.savefig(figure_root / f"{name}-boxplots-rollout-{signal_name}-{system_name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(figure_root / f"{name}-boxplots-rollout-{signal_name}-{system_name}.pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)
# print("Figures saved to:", figure_root)

### Validation horizons:

compute the errors for horizon = 29 and 100

In [ ]:
from matplotlib.patches import Patch


def rollout_prefix_mse(rmse_by_step, horizon):
    """Mean squared error over predicted steps 1..h, as in validation."""
    values = np.asarray(rmse_by_step, dtype=float)
    if values.ndim != 1 or not 1 <= horizon <= values.size:
        raise ValueError(f"Need at least {horizon} per-step RMSE values; got {values.shape}.")
    # Each step averages over the same trajectories and features.
    # Square before averaging: mean(RMSE) ** 2 is a different statistic.
    return float(np.mean(np.square(values[:horizon])))


horizons = (29, 49, 99)
prefix, suffix = "rollout.", ".metrics.rmse_by_step"
signals = sorted(
    column[len(prefix):-len(suffix)]
    for column in modeling_results.columns
    if column.startswith(prefix) and column.endswith(suffix)
)
if not signals:
    raise ValueError("No saved per-step rollout RMSE values found.")

metadata = ["exp_id", "system_name", "model_type", "excitation_objective"]


horizon_records = []
for _, run in modeling_results.iterrows():
    for signal in signals:

        rmse_by_step = run[f"rollout.{signal}.metrics.rmse_by_step"]
        if rmse_by_step is None or (
            np.ndim(rmse_by_step) == 0 and pd.isna(rmse_by_step)
        ):
            continue

        for horizon in horizons:
            mse = rollout_prefix_mse(rmse_by_step, horizon)
            horizon_records.append({
                **run[metadata].to_dict(),
                "signal": signal, "horizon": horizon, "mse": mse, "rmse": np.sqrt(mse),
            })


horizon_results = pd.DataFrame(horizon_records)

figure_root = QUANTITATIVE_RESULTS_ROOT / "modeling_figs" / "boxplots"
figure_root.mkdir(parents=True, exist_ok=True)
horizon_results.to_csv(figure_root / f"{name}-rollout-horizon-metrics.csv", index=False)

objectives = sorted(horizon_results["excitation_objective"].dropna().unique())
colors = {objective: plt.get_cmap("tab10")(i % 10) for i, objective in enumerate(objectives)}
for system_name, system_results in horizon_results.groupby("system_name", sort=True):
    model_types = sorted(system_results["model_type"].dropna().unique())
    system_objectives = [o for o in objectives if o in system_results["excitation_objective"].values]
    box_width = 0.8 / len(system_objectives)
    for horizon in horizons:
        fig, axes = plt.subplots(
            1, len(signals), figsize=(figurewidth_in, 3.5),
            squeeze=False, sharey=True, constrained_layout=True,
        )
        for ax, signal in zip(axes[0], signals):
            for model_index, model_type in enumerate(model_types):
                for objective_index, objective in enumerate(system_objectives):
                    values = system_results.loc[
                        (system_results["horizon"] == horizon)
                        & (system_results["signal"] == signal)
                        & (system_results["model_type"] == model_type)
                        & (system_results["excitation_objective"] == objective),
                        "mse",
                    ].to_numpy(dtype=float)
                    values = values[np.isfinite(values)]
                    if np.any(values <= 0):
                        raise ValueError("Logarithmic MSE plots require positive values.")
                    position = model_index + (objective_index - (len(system_objectives) - 1) / 2) * box_width
                    color = colors[objective]
                    if values.size:
                        ax.boxplot(
                            [values], positions=[position], widths=box_width * 0.8,
                            patch_artist=True, manage_ticks=False, showfliers=True,
                            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.7},
                            whiskerprops={"color": color}, capprops={"color": color},
                            medianprops={"color": "black"},
                            flierprops={"marker": ".", "markeredgecolor": color},
                        )
                    ax.text(
                        position, 0.98, f"n={values.size}", transform=ax.get_xaxis_transform(),
                        ha="center", va="top", fontsize=8, color=color,
                    )
            ax.set_title(signal.replace("_", " "))
            ax.set_yscale("log")
            ax.set_xlabel("Model type")
            ax.set_xticks(range(len(model_types)), [m.replace("_", " ") for m in model_types])
            ax.set_xlim(-0.5, len(model_types) - 0.5)
            ax.margins(y=0.15)
            ax.grid(True, which="major", alpha=0.3)
            ax.set_axisbelow(True)
        axes[0, 0].set_ylabel("MSE (log scale)")
        fig.suptitle(f"{system_name.replace('_', ' ').title()} — rollout h{horizon} (steps 1–{horizon})")
        fig.legend(
            handles=[Patch(facecolor=colors[o], alpha=0.7, label=o.replace("_", " ")) for o in system_objectives],
            title="Excitation objective", loc="lower center", bbox_to_anchor=(0.5, -0.15),
            ncol=len(system_objectives),
        )
        stem = f"{name}-boxplots-rollout-h{horizon}-{system_name}"
        fig.savefig(figure_root / f"{stem}.png", dpi=200, bbox_inches="tight")
        fig.savefig(figure_root / f"{stem}.pdf", bbox_inches="tight")
        display(fig)
        plt.close(fig)

print("Horizon metrics and figures saved to:", figure_root)